# Flow and Process Visualization (Solution)

Use Sankey, alluvial-style, and chord-like diagrams for transitions/flows.
Muc tieu: biet khi nao nen/khong nen dung tung loai chart.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go

sns.set_theme(style="whitegrid")

def resolve_repo_root() -> Path:
    cwd = Path.cwd().resolve()
    for p in [cwd, *cwd.parents]:
        if (p / "data" / "gapminder.csv").exists():
            return p
    raise FileNotFoundError("Cannot locate data/gapminder.csv")

root = resolve_repo_root()
df = pd.read_csv(root / "data" / "gapminder.csv")
d2007 = df[df["year"] == 2007].copy()


## 1) Sankey (toy flow)

In [ ]:
labels = ['Africa','Americas','Asia','Europe','Oceania','<60 lifeExp','60-75','>75']
source = [0,1,2,3,4,0,1,2,3,4]
target = [5,6,6,7,7,6,7,7,7,7]
value =  [20,15,25,18,5,10,12,9,6,2]
fig = go.Figure(data=[go.Sankey(node=dict(label=labels), link=dict(source=source, target=target, value=value))])
fig.update_layout(title_text='Toy Sankey: region -> lifeExp band')
fig.show()

## 2) Alluvial-like stacked area (approx)

In [ ]:
trend = df.groupby(['year','continent'],as_index=False)['pop'].sum()
p = trend.pivot(index='year', columns='continent', values='pop').fillna(0)
plt.figure(figsize=(9,4))
plt.stackplot(p.index, [p[c] for c in p.columns], labels=p.columns, alpha=0.8)
plt.legend(ncol=3, fontsize=8)
plt.title('Stacked area: population flow over time (continent)')
plt.tight_layout(); plt.show()

In [ ]:
# Toy OD matrix between regions, then render as circular network (chord-like surrogate)
import networkx as nx

regions = ["Africa", "Americas", "Asia", "Europe", "Oceania"]
flow_mat = pd.DataFrame(
    [
        [0, 5, 18, 4, 1],
        [4, 0, 12, 9, 2],
        [16, 10, 0, 14, 3],
        [3, 8, 11, 0, 2],
        [1, 2, 4, 3, 0],
    ],
    index=regions,
    columns=regions,
)

G = nx.DiGraph()
for src in regions:
    for dst in regions:
        w = flow_mat.loc[src, dst]
        if src != dst and w > 0:
            G.add_edge(src, dst, weight=w)

pos = nx.circular_layout(G)
weights = [G[u][v]["weight"] for u, v in G.edges()]
widths = [0.5 + w / 6 for w in weights]

plt.figure(figsize=(7, 7))
nx.draw_networkx_nodes(G, pos, node_size=2200, node_color="#f2f2f2", edgecolors="#333")
nx.draw_networkx_labels(G, pos, font_size=10)
nx.draw_networkx_edges(
    G,
    pos,
    width=widths,
    alpha=0.55,
    arrows=True,
    arrowstyle="-|>",
    arrowsize=12,
    connectionstyle="arc3,rad=0.12",
)
plt.title("Chord-like circular flow view (directed)")
plt.axis("off")
plt.tight_layout()
plt.show()

flow_mat

## 4) Chord-style ribbon approximation (Plotly)

Phan nay tao chord-style bang ribbon path de minh hoa ket noi hai chieu giua cac nhom.
Day la ban xap xi de giang day (de doc pattern), khong phai implementation chord engine day du.

In [ ]:
import math

# symmetric matrix for chord-style demo
nodes = ["Africa", "Americas", "Asia", "Europe", "Oceania"]
M = np.array([
    [0, 12, 28, 10, 4],
    [12, 0, 24, 18, 6],
    [28, 24, 0, 22, 8],
    [10, 18, 22, 0, 5],
    [4, 6, 8, 5, 0],
], dtype=float)

# angles on unit circle
n = len(nodes)
angles = np.linspace(0, 2 * np.pi, n, endpoint=False)
R = 1.0
node_xy = {nodes[i]: (R * np.cos(angles[i]), R * np.sin(angles[i])) for i in range(n)}

# helper: quadratic bezier for smooth ribbons

def qbez(p0, p1, pc, t):
    return (
        (1 - t) ** 2 * p0[0] + 2 * (1 - t) * t * pc[0] + t**2 * p1[0],
        (1 - t) ** 2 * p0[1] + 2 * (1 - t) * t * pc[1] + t**2 * p1[1],
    )

fig = go.Figure()

# draw outer circle
th = np.linspace(0, 2 * np.pi, 400)
fig.add_trace(
    go.Scatter(
        x=np.cos(th),
        y=np.sin(th),
        mode="lines",
        line=dict(color="#999", width=1),
        hoverinfo="skip",
        showlegend=False,
    )
)

# draw ribbons for upper triangle
max_w = M.max()
for i in range(n):
    for j in range(i + 1, n):
        w = M[i, j]
        if w <= 0:
            continue

        p0 = node_xy[nodes[i]]
        p1 = node_xy[nodes[j]]
        # control point closer to center; tweak by weight
        shrink = 0.22 + 0.18 * (w / max_w)
        pc = (0.0, 0.0)

        ts = np.linspace(0, 1, 60)
        curve = np.array([qbez(p0, p1, pc, t) for t in ts])

        fig.add_trace(
            go.Scatter(
                x=curve[:, 0],
                y=curve[:, 1],
                mode="lines",
                line=dict(width=1 + 9 * (w / max_w), color="rgba(44,127,184,0.35)"),
                hovertemplate=f"{nodes[i]} ↔ {nodes[j]}<br>flow={w}<extra></extra>",
                showlegend=False,
            )
        )

# draw nodes + labels
for name, (x, y) in node_xy.items():
    fig.add_trace(
        go.Scatter(
            x=[x],
            y=[y],
            mode="markers+text",
            marker=dict(size=14, color="#d62728"),
            text=[name],
            textposition="top center",
            hoverinfo="text",
            showlegend=False,
        )
    )

fig.update_layout(
    title="Chord-style ribbon approximation (symmetric inter-region flows)",
    xaxis=dict(visible=False),
    yaxis=dict(visible=False, scaleanchor="x", scaleratio=1),
    width=760,
    height=700,
    plot_bgcolor="white",
    paper_bgcolor="white",
)
fig.show()

## 3) Chord-like flow matrix (network-style surrogate)

Plotly không có chord chart native ổn định như Sankey, nên dùng ma trận flow + network circular như một cách thay thế để học tư duy chord.

## Reflection
- Nêu 2 điểm học được về chart selection.
- Chỉ ra 1 rủi ro diễn giải sai với loại chart trong lab này.

## Khi nào nên / không nên dùng
- **Sankey**: Nên dùng khi có luồng có hướng (A -> B -> C), cần thấy conservation của flow; không nên dùng khi node quá nhiều làm rối.
- **Alluvial-like**: Nên dùng cho composition chuyển theo thời gian/cohort; không nên dùng khi người xem cần đọc giá trị chính xác từng đường nhỏ.
- **Chord (hoặc chord-like)**: Nên dùng cho quan hệ hai chiều giữa nhóm (OD matrix dày), nhấn vào pattern kết nối; không nên dùng khi người xem cần đọc exact value từng cặp hoặc số nhóm quá lớn.